# Contextual retrieval

Same seeded chunks, same questions, four stages. Pass@k is 1 when the golden chunk is inside the top k.

1. Basic dense search embeds the raw chunk.
2. Contextual dense search embeds the situating sentence, then the chunk.
3. Contextual BM25 searches that same text and blends it with dense search. Dense weight 0.8, BM25 weight 0.2.
4. Rerank takes a wide contextual candidate list and keeps the top k.

The first cell indexes 130 documents: every knowledge brief, every incident, and a set of alerts. Pass@20 is then a small slice of the corpus. A tqdm bar shows each situating call.

Scored questions ask how to prevent a knowledge topic. The golden chunk is that document's Prevention section, which does not repeat the title. Basic dense search only sees that section. Contextual search and BM25 also see the document title. Incidents and alerts stay in the index as other candidates.

The last section calls the LangGraph agent. Status comes from SQLite. That section is not scored.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from security_investigator.config import Settings
from security_investigator.preprocess import (
    preprocess_corpus,
    sentence_transformer_embedder,
    vectors_for_chunks,
)
from security_investigator.retrieval import BM25Index, MemoryDenseIndex

DOCUMENT_LIMIT = 130

settings = Settings(_env_file=ROOT / ".env")
database = ROOT / settings.database_path
index_directory = ROOT / "data" / "indexes" / "sample"
if not database.exists():
    raise SystemExit("database is missing; run scripts/seed_database.py first")

built = await preprocess_corpus(
    database,
    index_directory,
    settings=settings,
    document_limit=DOCUMENT_LIMIT,
    progress=True,
)
chunks = built["chunks"]
basic_vectors = built["basic_embeddings"]
contextual_vectors = built["contextual_embeddings"]

print(f"{len(chunks)} chunks, {sum(1 for chunk in chunks if chunk.get('context'))} with context")

In [ ]:
def questions_from_chunks(records: list[dict]) -> list[dict]:
    """One question per knowledge document.

    The golden chunk is the Prevention section. The question names the document,
    and that name is not in the section text.
    """
    grouped: dict[str, list[dict]] = {}
    for chunk in records:
        grouped.setdefault(chunk["document_id"], []).append(chunk)
    questions = []
    for doc_chunks in grouped.values():
        if doc_chunks[0]["document_type"] != "security_knowledge":
            continue
        golden = doc_chunks[-1]
        title = str(golden["title"])
        if title.lower() in str(golden["text"]).lower():
            continue
        questions.append(
            {
                "query": f"How do you prevent {title}?",
                "golden_chunk_id": golden["chunk_id"],
            }
        )
    return questions


def make_index(records: list[dict], embeddings: list[list[float]]) -> MemoryDenseIndex:
    index = MemoryDenseIndex()
    grouped = vectors_for_chunks(
        records, embeddings, dimension=settings.embedding_dimension
    )
    for namespace, vectors in grouped.items():
        index.upsert(namespace, vectors)
    return index


questions = questions_from_chunks(chunks)
embedder = sentence_transformer_embedder(settings.embedding_model)
basic_index = make_index(chunks, basic_vectors)
contextual_index = make_index(chunks, contextual_vectors)
bm25 = BM25Index(chunks)
print(f"{len(questions)} questions, {len(chunks)} chunks in the index")

In [ ]:
from tqdm.auto import tqdm

from security_investigator.retrieval import dense_search, hybrid_search, rerank

KS = (5, 10, 20)
WIDE_K = min(len(chunks), 100)


async def basic_ids(query: str, k: int) -> list[str]:
    hits = await dense_search(
        query,
        top_k=k,
        settings=settings,
        embedder=embedder,
        index=basic_index,
        chunks=chunks,
    )
    return [hit.chunk_id for hit in hits]


async def contextual_ids(query: str, k: int) -> list[str]:
    hits = await dense_search(
        query,
        top_k=k,
        settings=settings,
        embedder=embedder,
        index=contextual_index,
        chunks=chunks,
    )
    return [hit.chunk_id for hit in hits]


async def hybrid_ids(query: str, k: int) -> list[str]:
    hits = await hybrid_search(
        query,
        top_k=k,
        settings=settings,
        embedder=embedder,
        index=contextual_index,
        chunks=chunks,
        bm25_index=bm25,
    )
    return [hit.chunk_id for hit in hits]


async def reranked_ids(query: str, k: int) -> list[str]:
    wide = await dense_search(
        query,
        top_k=WIDE_K,
        settings=settings,
        embedder=embedder,
        index=contextual_index,
        chunks=chunks,
    )
    ranked = rerank(query, wide, top_k=k)
    return [hit.chunk_id for hit in ranked]


async def pass_at(search, name: str) -> dict[int, float]:
    hits = {k: 0 for k in KS}
    for question in tqdm(questions, desc=name):
        ids = await search(question["query"], max(KS))
        for k in KS:
            if question["golden_chunk_id"] in ids[:k]:
                hits[k] += 1
    return {k: hits[k] / len(questions) for k in KS}


stages = [
    ("Basic dense", basic_ids),
    ("Contextual dense", contextual_ids),
    ("Contextual BM25 hybrid", hybrid_ids),
    ("Rerank", reranked_ids),
]
rows = []
for name, search in stages:
    scores = await pass_at(search, name)
    rows.append({"stage": name, **{f"pass_at_{k}": scores[k] for k in KS}})
    print(
        f"{name:24}  "
        + "  ".join(f"Pass@{k} {scores[k]:.3f}" for k in KS)
    )

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
labels = [row["stage"] for row in rows]
values = [row["pass_at_10"] for row in rows]
bars = ax.bar(labels, values, color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
ax.bar_label(bars, fmt="%.2f", padding=2)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Pass@10")
ax.set_title(f"Contextual retrieval on {len(questions)} seeded questions")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

## Investigation

The graph reads the contextual index, then takes incident status from SQLite. This cell prints that answer. It does not add a retrieval score.

In [ ]:
from security_investigator.agent import InvestigationDeps, investigate
from security_investigator.llm import answer_model_from_settings

result = await investigate(
    "Which critical incidents are active?",
    deps=InvestigationDeps(
        database_path=database,
        settings=settings,
        embedder=embedder,
        index=contextual_index,
        chunks=chunks,
        bm25_index=bm25,
        rerank_enabled=False,
        answer_model=answer_model_from_settings(settings),
    ),
)
print(result.answer)
for finding in result.findings:
    print(f"incident {finding.incident_id}: {finding.status} ({finding.severity})")
if result.unverified_alerts:
    print("unverified alerts:", result.unverified_alerts)
if result.limitations:
    print("limitations:", result.limitations)